# Chapter 30: Dynamic Programming II: Sequences, Strings and Knapsack

The programs from Chapter 30 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Longest Common Subsequence

`dp_sequences.py`

In [1]:
"""Dynamic programming on sequences: tables indexed by positions in one
or two sequences."""

import bisect


def lcs(a, b):
    """The longest common subsequence of a and b (in order, not
    necessarily side by side). table[i][j] is the LCS length of
    a[:i] and b[:j]; walk back through it to recover the letters."""
    table = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    out, i, j = [], len(a), len(b)
    while i and j:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1])
            i, j = i - 1, j - 1
        elif table[i - 1][j] >= table[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return out[::-1]


def edit_distance(a, b):
    """The fewest single-letter inserts, deletes and replacements that
    turn a into b. Only two rows of the table are kept."""
    prev = list(range(len(b) + 1))  # turning "" into b[:j]: j inserts
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                cur[j] = prev[j - 1]
            else:
                cur[j] = 1 + min(prev[j - 1], prev[j], cur[j - 1])
        prev = cur
    return prev[-1]


def knapsack(items, capacity):
    """The most value from (weight, value) items, each taken at most
    once, within capacity. best[c] is the best value using weight c;
    looping c downward keeps each item from being used twice."""
    best = [0] * (capacity + 1)
    for weight, value in items:
        for c in range(capacity, weight - 1, -1):
            best[c] = max(best[c], best[c - weight] + value)
    return best[capacity]


def lis_quadratic(nums):
    """Length of the longest strictly increasing subsequence, O(n^2):
    ends[i] is the longest one ending at nums[i]."""
    ends = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                ends[i] = max(ends[i], ends[j] + 1)
    return max(ends, default=0)


def lis(nums):
    """The same in O(n log n): tails[k] is the smallest possible last
    item of an increasing subsequence of length k + 1. It stays sorted,
    so each new number finds its place by binary search."""
    tails = []
    for x in nums:
        k = bisect.bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)

`dp_sequences_demo.py`

In [2]:
"""The sequence DPs on small examples."""

from dp_sequences import edit_distance, knapsack, lcs, lis

print("".join(lcs("ABCBDAB", "BDCABA")))
before = ["import os", "x = 1", "y = 2", "print(x)"]
after = ["import os", "import sys", "x = 1", "print(x)"]
print(lcs(before, after))  # the lines a diff keeps
print(edit_distance("kitten", "sitting"), edit_distance("flaw", "lawn"))
items = [(1, 1), (3, 4), (4, 5), (5, 7)]  # (weight, value)
print(knapsack(items, 7))
print(lis([10, 9, 2, 5, 3, 7, 101, 18]))

BCBA
['import os', 'x = 1', 'print(x)']
3 2
9
4


## Edit Distance

`edit_table.py`

In [3]:
"""The full edit-distance table for kitten -> sitting. Each cell is the
distance between a prefix of one word and a prefix of the other."""

a, b = "kitten", "sitting"
table = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
for i in range(len(a) + 1):
    for j in range(len(b) + 1):
        if i == 0 or j == 0:
            table[i][j] = i + j
        elif a[i - 1] == b[j - 1]:
            table[i][j] = table[i - 1][j - 1]
        else:
            table[i][j] = 1 + min(
                table[i - 1][j - 1], table[i - 1][j], table[i][j - 1]
            )
print("      " + "  ".join(b))
for i, row in enumerate(table):
    label = a[i - 1] if i else " "
    print(f"{label}  " + "  ".join(map(str, row)))

      s  i  t  t  i  n  g
   0  1  2  3  4  5  6  7
k  1  1  2  3  4  5  6  7
i  2  2  1  2  3  4  5  6
t  3  3  2  1  2  3  4  5
t  4  4  3  2  1  2  3  4
e  5  5  4  3  2  2  3  4
n  6  6  5  4  3  3  2  3


## Timing

`bench_lis.py`

In [4]:
"""Longest increasing subsequence: O(n^2) against O(n log n)."""

import random

import bench
from dp_sequences import lis, lis_quadratic

rng = random.Random(2026)


def make(n):
    return ([rng.random() for _ in range(n)],)


sizes = [500, 1_000, 2_000, 4_000]
results = {
    "n squared": bench.doubling(lis_quadratic, make, sizes),
    "n log n": bench.doubling(lis, make, sizes),
}
bench.table(results)
bench.chart(results, "ch30_lis.png", "Longest increasing subsequence")

         n       n squared         n log n
       500         5.07 ms         37.1 us
     1,000      21 ms x4.1    89.7 us x2.4
     2,000    86.6 ms x4.1     202 us x2.3
     4,000     349 ms x4.0     429 us x2.1
